# Cell 01 - Load corpus chunks để chuẩn bị build BM25 index

## Mục tiêu cell này
Đọc file `all_chunks.csv` đã tạo ở notebook 03 để chuẩn bị xây dựng BM25 index.

## Vì sao cần làm bước này?
BM25 là phương pháp truy xuất dựa trên từ khóa.  
Trong hệ thống RAG của project, BM25 sẽ giúp tìm các chunk có nhiều từ khóa trùng hoặc gần trùng với câu hỏi người dùng.

Ví dụ người dùng hỏi:
"Công ty có chính sách gì về thiết bị làm việc?"

BM25 có thể tìm các chunk chứa từ khóa như:
- work devices
- managing work devices
- company equipment
- device policy

BM25 sẽ là:
- baseline retrieval đầu tiên
- một nhánh trong Hybrid Retrieval cùng với Dense Retrieval
- nguồn kết quả để sau này kết hợp bằng RRF và reranker

## Giải thích code
Code sẽ:
1. Xác định đúng thư mục project
2. Đọc `data/chunks/all_chunks.csv`
3. Tạo thư mục lưu BM25 index
4. Kiểm tra số lượng chunk và phân bố nguồn dữ liệu
5. Hiển thị vài dòng đầu để xác nhận corpus đã sẵn sàng

## Output mong đợi
Bạn cần thấy:
- tổng số chunk khoảng 91,448
- `source_type` gồm `legal` và `company_handbook`
- bảng preview có các cột `corpus_id`, `chunk_id`, `parent_id`, `source_type`, `title`

In [1]:
from pathlib import Path
import pandas as pd
import re
import pickle

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

CHUNK_DIR = PROJECT / "data" / "chunks"
BM25_DIR = PROJECT / "indexes" / "bm25"

BM25_DIR.mkdir(parents=True, exist_ok=True)

all_chunks_df = pd.read_csv(CHUNK_DIR / "all_chunks.csv")

print("Project:", PROJECT)
print("All chunks shape:", all_chunks_df.shape)

print("\nSource distribution:")
display(all_chunks_df["source_type"].value_counts().reset_index())

display(
    all_chunks_df[
        ["corpus_id", "chunk_id", "parent_id", "source_type", "title", "chunk_chars", "language"]
    ].head()
)

Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
All chunks shape: (91448, 11)

Source distribution:


C:\Users\npd20\AppData\Local\Temp\ipykernel_23324\572102667.py:14: DtypeWarning: Columns (2,5) have mixed types. Specify dtype option on import or set low_memory=False.
  all_chunks_df = pd.read_csv(CHUNK_DIR / "all_chunks.csv")


,source_type,count
0,legal,91353
1,company_handbook,95


,corpus_id,chunk_id,parent_id,source_type,title,chunk_chars,language
0,chunk_000000,legal_142820_000,142820,legal,legal_cid_142820,767,Vietnamese
1,chunk_000001,legal_27817_000,27817,legal,legal_cid_27817,1194,Vietnamese
2,chunk_000002,legal_27817_001,27817,legal,legal_cid_27817,212,Vietnamese
3,chunk_000003,legal_72117_000,72117,legal,legal_cid_72117,811,Vietnamese
4,chunk_000004,legal_33215_000,33215,legal,legal_cid_33215,394,Vietnamese


# Cell 02 - Chuẩn hóa kiểu dữ liệu metadata trước khi build BM25

## Mục tiêu cell này
Chuẩn hóa các cột metadata trong `all_chunks_df` để tránh lỗi kiểu dữ liệu khi lưu index và truy xuất kết quả BM25.

## Vì sao cần làm bước này?
Trong corpus chung, `parent_id` của legal là số, còn `parent_id` của company handbook là chuỗi như `company_0000`.  
Vì vậy pandas cảnh báo `DtypeWarning` do cột có kiểu dữ liệu trộn.

Nếu không chuẩn hóa, các bước sau có thể gặp lỗi khi:
- lưu metadata
- lọc theo `parent_id`
- so sánh kết quả retrieval với `relevant_cids`
- hiển thị nguồn trong demo

## Giải thích code
Code sẽ:
1. Đọc lại `all_chunks.csv` với `low_memory=False`
2. Ép các cột định danh về dạng chuỗi
3. Ép `chunk_chars` và `chunk_index` về số
4. Kiểm tra lại số dòng và kiểu dữ liệu sau khi chuẩn hóa

## Output mong đợi
Bạn cần thấy corpus vẫn có 91,448 dòng và các cột định danh như `corpus_id`, `chunk_id`, `parent_id` có kiểu `object`.

In [2]:
all_chunks_df = pd.read_csv(CHUNK_DIR / "all_chunks.csv", low_memory=False)

id_cols = ["corpus_id", "chunk_id", "parent_id", "source_type", "title", "source_path", "language"]

for col in id_cols:
    all_chunks_df[col] = all_chunks_df[col].fillna("").astype(str)

all_chunks_df["retrieval_text"] = all_chunks_df["retrieval_text"].fillna("").astype(str)
all_chunks_df["chunk_text"] = all_chunks_df["chunk_text"].fillna("").astype(str)
all_chunks_df["chunk_chars"] = pd.to_numeric(all_chunks_df["chunk_chars"], errors="coerce").fillna(0).astype(int)
all_chunks_df["chunk_index"] = pd.to_numeric(all_chunks_df["chunk_index"], errors="coerce").fillna(0).astype(int)

print("All chunks shape:", all_chunks_df.shape)
print("\nDtypes:")
display(all_chunks_df.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"}))

display(all_chunks_df[["corpus_id", "chunk_id", "parent_id", "source_type", "title", "chunk_chars"]].head())

All chunks shape: (91448, 11)

Dtypes:


,column,dtype
0,corpus_id,object
1,chunk_id,object
2,parent_id,object
3,source_type,object
4,title,object
5,source_path,object
6,chunk_index,int64
7,chunk_text,object
8,chunk_chars,int64
9,language,object


,corpus_id,chunk_id,parent_id,source_type,title,chunk_chars
0,chunk_000000,legal_142820_000,142820,legal,legal_cid_142820,767
1,chunk_000001,legal_27817_000,27817,legal,legal_cid_27817,1194
2,chunk_000002,legal_27817_001,27817,legal,legal_cid_27817,212
3,chunk_000003,legal_72117_000,72117,legal,legal_cid_72117,811
4,chunk_000004,legal_33215_000,33215,legal,legal_cid_33215,394


# Cell 03 - Tạo hàm tokenize cho BM25

## Mục tiêu cell này
Tạo hàm tách từ đơn giản để chuẩn bị dữ liệu đầu vào cho BM25.

## Vì sao cần làm bước này?
BM25 không làm việc trực tiếp trên câu văn thô.  
Nó cần mỗi tài liệu được chuyển thành danh sách token.

Ví dụ:

Câu hỏi:
`Công ty có chính sách gì về thiết bị làm việc?`

Sau tokenize:
`["công", "ty", "có", "chính", "sách", "gì", "về", "thiết", "bị", "làm", "việc"]`

Với project này, tokenizer cần dùng được cho cả:
- văn bản pháp luật tiếng Việt
- handbook nội bộ tiếng Anh

## Giải thích code
Code sẽ:
1. Chuyển văn bản về chữ thường
2. Lấy các cụm chữ/số bằng regex
3. Loại token quá ngắn nếu cần
4. Test thử trên 2 câu: một câu tiếng Việt và một câu tiếng Anh

Tokenizer này là bản baseline cho BM25. Sau này nếu muốn cải thiện, có thể thay bằng tokenizer tiếng Việt như `underthesea` hoặc `pyvi`.

## Output mong đợi
Bạn cần thấy danh sách token của câu tiếng Việt và tiếng Anh.

In [3]:
def bm25_tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-zA-ZÀ-ỹ0-9]+", text)
    return [tok for tok in tokens if len(tok) > 1]

sample_vi = "Công ty có chính sách gì về thiết bị làm việc?"
sample_en = "What is the company policy for managing work devices?"

print("Vietnamese tokens:")
print(bm25_tokenize(sample_vi))

print("\nEnglish tokens:")
print(bm25_tokenize(sample_en))

Vietnamese tokens:
['công', 'ty', 'có', 'chính', 'sách', 'gì', 'về', 'thiết', 'bị', 'làm', 'việc']

English tokens:
['what', 'is', 'the', 'company', 'policy', 'for', 'managing', 'work', 'devices']


# Cell 04 - Tokenize toàn bộ corpus cho BM25

## Mục tiêu cell này
Chuyển toàn bộ `retrieval_text` trong `all_chunks_df` thành danh sách token để xây dựng BM25 index.

## Vì sao cần làm bước này?
BM25 cần corpus ở dạng danh sách token, không phải văn bản thô.

Ví dụ:
- Một chunk pháp luật tiếng Việt sẽ được tách thành các token pháp lý.
- Một chunk handbook tiếng Anh sẽ được tách thành các token chính sách nội bộ.

Sau bước này, mỗi dòng chunk sẽ có một danh sách token tương ứng. Đây là dữ liệu đầu vào trực tiếp cho `BM25Okapi`.

## Giải thích code
Code sẽ:
1. Lấy cột `retrieval_text`
2. Áp dụng hàm `bm25_tokenize()` cho từng chunk
3. Tạo biến `tokenized_corpus`
4. Kiểm tra số lượng tài liệu đã tokenize
5. Hiển thị thử token của chunk đầu tiên

## Output mong đợi
Bạn cần thấy:
- số tài liệu tokenized bằng 91,448
- token mẫu của chunk đầu tiên

In [4]:
tokenized_corpus = all_chunks_df["retrieval_text"].apply(bm25_tokenize).tolist()

print("Số chunks:", len(all_chunks_df))
print("Số tokenized documents:", len(tokenized_corpus))

print("\nToken mẫu của chunk đầu tiên:")
print(tokenized_corpus[0][:50])

print("\nĐộ dài token trung bình:")
avg_tokens = sum(len(x) for x in tokenized_corpus) / len(tokenized_corpus)
print(round(avg_tokens, 2))

Số chunks: 91448
Số tokenized documents: 91448

Token mẫu của chunk đầu tiên:
['legal', 'cid', '142820', 'điều', 'địa', 'vị', 'pháp', 'lý', 'của', 'liên', 'đoàn', 'luật', 'sư', 'việt', 'nam', 'liên', 'đoàn', 'luật', 'sư', 'việt', 'nam', 'là', 'tổ', 'chức', 'xã', 'hội', 'nghề', 'nghiệp', 'thống', 'nhất', 'trong', 'toàn', 'quốc', 'của', 'các', 'đoàn', 'luật', 'sư', 'các', 'luật', 'sư', 'việt', 'nam', 'có', 'tư', 'cách', 'pháp', 'nhân', 'có', 'con']

Độ dài token trung bình:
162.62


# Cell 05 - Build BM25 index từ tokenized corpus

## Mục tiêu cell này
Xây dựng BM25 index từ toàn bộ corpus đã tokenize.

## Vì sao cần làm bước này?
BM25 là bộ truy xuất theo từ khóa.  
Sau khi build xong, hệ thống có thể nhận một câu hỏi người dùng, tokenize câu hỏi đó, rồi tìm ra các chunk có từ khóa liên quan nhất.

Ví dụ:
- Query: `thiết bị làm việc`
- BM25 sẽ ưu tiên các chunk có nhiều từ như `thiết`, `bị`, `làm`, `việc`, hoặc tiếng Anh như `work devices` nếu query phù hợp.

BM25 sẽ được dùng cho:
- baseline keyword retrieval
- một nhánh trong Hybrid RAG
- so sánh với Dense Retrieval ở các notebook sau

## Giải thích code
Code sẽ:
1. Import `BM25Okapi` từ thư viện `rank_bm25`
2. Build BM25 index từ `tokenized_corpus`
3. Lưu BM25 index vào `indexes/bm25/bm25_index.pkl`
4. Lưu metadata chunks vào `indexes/bm25/bm25_metadata.csv`
5. In thông tin xác nhận index đã được tạo

## Output mong đợi
Bạn cần thấy:
- số document trong BM25 index bằng 91,448
- file BM25 index được lưu thành công
- file metadata được lưu thành công

In [6]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "rank-bm25"])

print("Cài rank-bm25 xong")

Cài rank-bm25 xong


In [7]:
from rank_bm25 import BM25Okapi
import pickle

bm25_index = BM25Okapi(tokenized_corpus)

index_path = BM25_DIR / "bm25_index.pkl"
metadata_path = BM25_DIR / "bm25_metadata.csv"

with open(index_path, "wb") as f:
    pickle.dump(bm25_index, f)

all_chunks_df.to_csv(metadata_path, index=False, encoding="utf-8-sig")

print("BM25 documents:", len(tokenized_corpus))
print("Đã lưu BM25 index:", index_path)
print("Đã lưu BM25 metadata:", metadata_path)
print("Index size MB:", round(index_path.stat().st_size / 1024 / 1024, 2))
print("Metadata size MB:", round(metadata_path.stat().st_size / 1024 / 1024, 2))

BM25 documents: 91448
Đã lưu BM25 index: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\indexes\bm25\bm25_index.pkl
Đã lưu BM25 metadata: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\indexes\bm25\bm25_metadata.csv
Index size MB: 71.46
Metadata size MB: 182.62


# Cell 06 - Tạo hàm BM25 search và kiểm tra truy xuất thử

## Mục tiêu cell này
Tạo hàm tìm kiếm bằng BM25 và chạy thử trên một vài câu hỏi mẫu.

## Vì sao cần làm bước này?
Sau khi build BM25 index, ta cần kiểm tra index có hoạt động đúng không.  
Nếu BM25 trả về các chunk liên quan, ta có thể dùng nó làm:
- baseline retrieval
- nhánh keyword retrieval trong Hybrid RAG
- dữ liệu đầu vào cho RRF và reranker sau này

## Lưu ý quan trọng
BM25 là tìm kiếm theo từ khóa, nên:
- Query tiếng Việt sẽ tìm tốt trong legal corpus tiếng Việt.
- Query tiếng Anh sẽ tìm tốt trong company handbook tiếng Anh.
- Query tiếng Việt hỏi handbook tiếng Anh có thể yếu, vì khác ngôn ngữ. Phần này sau sẽ cải thiện bằng Dense Retrieval multilingual.

## Giải thích code
Code sẽ:
1. Định nghĩa hàm `bm25_search()`
2. Tokenize câu hỏi bằng `bm25_tokenize()`
3. Tính BM25 score cho toàn bộ corpus
4. Lấy top-k chunk có điểm cao nhất
5. Test một câu hỏi pháp luật tiếng Việt và một câu hỏi handbook tiếng Anh

## Output mong đợi
Bạn cần thấy bảng kết quả gồm:
- `rank`
- `score`
- `source_type`
- `title`
- `chunk_text`

Kết quả legal query nên trả về các chunk pháp luật.  
Kết quả company query nên trả về các chunk handbook nội bộ.

In [8]:
import numpy as np

def bm25_search(query, top_k=5, source_filter=None):
    query_tokens = bm25_tokenize(query)
    scores = bm25_index.get_scores(query_tokens)

    if source_filter is not None:
        candidate_indices = all_chunks_df.index[
            all_chunks_df["source_type"].isin(source_filter)
        ].to_numpy()
    else:
        candidate_indices = np.arange(len(all_chunks_df))

    candidate_scores = scores[candidate_indices]
    top_local = np.argsort(candidate_scores)[::-1][:top_k]
    top_indices = candidate_indices[top_local]

    results = all_chunks_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results["rank"] = range(1, len(results) + 1)

    return results[
        [
            "rank",
            "score",
            "corpus_id",
            "chunk_id",
            "parent_id",
            "source_type",
            "title",
            "language",
            "chunk_text"
        ]
    ]

legal_query = "Lương thử việc được quy định như thế nào?"
company_query = "What is the company policy for managing work devices?"

print("BM25 test - Legal query:")
display(bm25_search(legal_query, top_k=5, source_filter=["legal"]))

print("\nBM25 test - Company handbook query:")
display(bm25_search(company_query, top_k=5, source_filter=["company_handbook"]))

BM25 test - Legal query:


,rank,score,corpus_id,chunk_id,parent_id,source_type,title,language,chunk_text
6876,1,28.138447,chunk_006876,legal_106663_000,106663,legal,legal_cid_106663,Vietnamese,"""1. Quy định mức lương tối thiểu tháng và mức ..."
67233,2,27.446146,chunk_067233,legal_215579_000,215579,legal,legal_cid_215579,Vietnamese,"""Điều 3. Mức lương tối thiểu\n3. Việc áp dụng ..."
14765,3,27.147801,chunk_014765,legal_75051_001,75051,legal,legal_cid_75051,Vietnamese,"định;\n- Thời gian bị đình chỉ công tác, bị tạ..."
59365,4,26.905449,chunk_059365,legal_118025_000,118025,legal,legal_cid_118025,Vietnamese,Về trả lương khi NLĐ đi làm vào những ngày ngh...
47527,5,26.775181,chunk_047527,legal_172654_000,172654,legal,legal_cid_172654,Vietnamese,Về các chế độ bảo hiểm đối với người thử việc ...



BM25 test - Company handbook query:


,rank,score,corpus_id,chunk_id,parent_id,source_type,title,language,chunk_text
91394,1,77.406124,chunk_091394,company_0008_000,company_0008,company_handbook,Readme,English,Read on basecamp.com\n\n# 37signals Employee H...
91382,2,73.252196,chunk_091382,company_0004_000,company_0004,company_handbook,Managing Work Devices,English,# Managing work devices\nEveryone receives a n...
91390,3,68.795058,chunk_091390,company_0006_001,company_0006,company_handbook,Our Internal Systems,English,## Grafana\nWe monitor our systems and their h...
91370,4,64.372216,chunk_091370,company_0002_000,company_0002,company_handbook,How We Work,English,# How We Work\n## Remotely\n37signals is a ful...
91387,5,60.954086,chunk_091387,company_0005_002,company_0005,company_handbook,Moonlighting,English,## Not OK\n1. You can’t work full time or part...


# Cell 07 - Load validation/test split để đánh giá BM25 Retrieval

## Mục tiêu cell này
Đọc lại `val_strict.csv` và `test_final.csv` để chuẩn bị đánh giá chất lượng truy xuất của BM25.

## Vì sao cần làm bước này?
BM25 đã build xong và đã search thử thủ công.  
Nhưng để biết BM25 tốt hay yếu, ta cần đánh giá bằng metric chính thức trên validation set.

Các metric sẽ dùng ở các cell sau:
- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR
- nDCG@10

Validation set dùng để kiểm tra và chọn tham số.  
Test set chỉ dùng ở bước đánh giá cuối, không dùng để chọn mô hình.

## Giải thích code
Code sẽ:
1. Load `val_strict.csv`
2. Load `test_final.csv`
3. Chuyển cột `relevant_cids` từ chuỗi JSON về list Python
4. Kiểm tra số dòng và hiển thị vài mẫu đầu tiên

## Output mong đợi
Bạn cần thấy:
- validation khoảng 8,605 dòng
- test khoảng 29,746 dòng
- cột `relevant_cids` hiển thị dạng list như `[243090]`

In [9]:
import json

SPLIT_DIR = PROJECT / "data" / "splits"

val_df = pd.read_csv(SPLIT_DIR / "val_strict.csv")
test_df = pd.read_csv(SPLIT_DIR / "test_final.csv")

val_df["relevant_cids"] = val_df["relevant_cids"].apply(json.loads)
test_df["relevant_cids"] = test_df["relevant_cids"].apply(json.loads)

print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(val_df.head())

Validation shape: (8605, 4)
Test shape: (29746, 4)


,qid,question,relevant_cids,num_relevant
0,162228,Chế độ báo cáo của Giám đốc Ban Quản lý dự án ...,[243090],1
1,3414,Việc tích nước vận hành và tích nước thời kỳ t...,[65231],1
2,108965,Phanh cần trục khi dừng khẩn cấp phải là loại ...,[183501],1
3,103161,Ai có trách nhiệm báo cáo kết quả đánh giá ngh...,[177010],1
4,70371,Làm sao để được công nhận văn bằng nước ngoài ...,"[140295, 140296]",2


# Cell 08 - Định nghĩa metric đánh giá Retrieval

## Mục tiêu cell này
Tạo các hàm đánh giá chất lượng truy xuất tài liệu cho BM25 Retrieval.

## Vì sao cần làm bước này?
Trong RAG, phần quan trọng đầu tiên là retriever có tìm đúng evidence hay không.  
Với dataset pháp luật Việt Nam, mỗi câu hỏi có `relevant_cids`, tức là danh sách `cid` đúng.

Khi BM25 trả về danh sách `parent_id`, ta so sánh với `relevant_cids` để tính:
- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR
- nDCG@10

Các metric này sẽ được dùng lại cho nhiều phiên bản:
- BM25 Retrieval
- Dense Retrieval
- Hybrid Retrieval
- Hybrid + Reranker

## Giải thích code
Code sẽ:
1. Tạo hàm `recall_at_k()`
2. Tạo hàm `mrr_score()`
3. Tạo hàm `ndcg_at_k()`
4. Tạo hàm `compute_retrieval_metrics()`
5. Test thử metric bằng một ví dụ nhỏ

## Output mong đợi
Bạn cần thấy metric mẫu chạy đúng, ví dụ nếu tài liệu đúng nằm ở vị trí thứ 2 thì Recall@3 = 1 và MRR = 0.5.

In [10]:
import math

def recall_at_k(pred_ids, relevant_ids, k):
    pred_top_k = pred_ids[:k]
    return 1.0 if set(pred_top_k) & set(relevant_ids) else 0.0

def mrr_score(pred_ids, relevant_ids):
    relevant_set = set(relevant_ids)
    for rank, pred_id in enumerate(pred_ids, start=1):
        if pred_id in relevant_set:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(pred_ids, relevant_ids, k=10):
    relevant_set = set(relevant_ids)
    dcg = 0.0

    for i, pred_id in enumerate(pred_ids[:k], start=1):
        rel = 1.0 if pred_id in relevant_set else 0.0
        dcg += rel / math.log2(i + 1)

    ideal_hits = min(len(relevant_set), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))

    return dcg / idcg if idcg > 0 else 0.0

def compute_retrieval_metrics(pred_ids, relevant_ids):
    return {
        "recall@1": recall_at_k(pred_ids, relevant_ids, 1),
        "recall@3": recall_at_k(pred_ids, relevant_ids, 3),
        "recall@5": recall_at_k(pred_ids, relevant_ids, 5),
        "recall@10": recall_at_k(pred_ids, relevant_ids, 10),
        "mrr": mrr_score(pred_ids, relevant_ids),
        "ndcg@10": ndcg_at_k(pred_ids, relevant_ids, 10)
    }

test_pred = [111, 222, 333, 444]
test_rel = [222]

print(compute_retrieval_metrics(test_pred, test_rel))

{'recall@1': 0.0, 'recall@3': 1.0, 'recall@5': 1.0, 'recall@10': 1.0, 'mrr': 0.5, 'ndcg@10': 0.6309297535714575}


# Cell 09 - Tạo hàm BM25 retrieve parent_id để đánh giá retrieval

## Mục tiêu cell này
Tạo hàm truy xuất BM25 trả về danh sách `parent_id` của các chunk được tìm thấy.

## Vì sao cần làm bước này?
BM25 thực tế trả về các chunk cụ thể, ví dụ:
`legal_27817_000`, `legal_27817_001`

Nhưng ground truth trong dataset lại nằm ở dạng `relevant_cids`, ví dụ:
`[27817]`

Vì vậy, khi đánh giá retrieval, ta cần quy kết quả từ `chunk_id` về `parent_id`.
Với legal corpus, `parent_id` chính là `cid`.

## Giải thích code
Code sẽ:
1. Tokenize câu hỏi
2. Tính BM25 score cho toàn bộ corpus
3. Lấy top-k chunk có điểm cao nhất
4. Lọc chỉ lấy `source_type = legal` để đánh giá trên legal benchmark
5. Trả về danh sách `parent_id` không trùng lặp theo thứ tự xếp hạng
6. Test thử trên 3 câu validation đầu tiên

## Output mong đợi
Bạn cần thấy mỗi câu validation có:
- question
- relevant_cids
- bm25_pred_parent_ids
- metrics

In [11]:
def bm25_retrieve_parent_ids(query, top_k=10, source_filter=["legal"], candidate_multiplier=5):
    query_tokens = bm25_tokenize(query)
    scores = bm25_index.get_scores(query_tokens)

    if source_filter is not None:
        candidate_indices = all_chunks_df.index[
            all_chunks_df["source_type"].isin(source_filter)
        ].to_numpy()
    else:
        candidate_indices = np.arange(len(all_chunks_df))

    candidate_scores = scores[candidate_indices]
    raw_k = min(top_k * candidate_multiplier, len(candidate_indices))

    top_local = np.argpartition(candidate_scores, -raw_k)[-raw_k:]
    top_local = top_local[np.argsort(candidate_scores[top_local])[::-1]]
    top_indices = candidate_indices[top_local]

    parent_ids = []
    seen = set()

    for idx in top_indices:
        parent_id = str(all_chunks_df.iloc[idx]["parent_id"])
        if parent_id not in seen:
            parent_ids.append(parent_id)
            seen.add(parent_id)
        if len(parent_ids) >= top_k:
            break

    return parent_ids

for i in range(3):
    row = val_df.iloc[i]
    pred_ids = bm25_retrieve_parent_ids(row["question"], top_k=10)
    relevant_ids = [str(x) for x in row["relevant_cids"]]
    metrics = compute_retrieval_metrics(pred_ids, relevant_ids)

    print("\n--- Sample", i, "---")
    print("Question:", row["question"])
    print("Relevant cids:", relevant_ids)
    print("BM25 pred parent_ids:", pred_ids[:10])
    print("Metrics:", metrics)


--- Sample 0 ---
Question: Chế độ báo cáo của Giám đốc Ban Quản lý dự án đầu tư xây dựng chuyên ngành thuộc Kiểm toán nhà nước được quy định như thế nào?
Relevant cids: ['243090']
BM25 pred parent_ids: ['104255', '220607', '243090', '98280', '115404', '169240', '214658', '129839', '169212', '23765']
Metrics: {'recall@1': 0.0, 'recall@3': 1.0, 'recall@5': 1.0, 'recall@10': 1.0, 'mrr': 0.3333333333333333, 'ndcg@10': 0.5}

--- Sample 1 ---
Question: Việc tích nước vận hành và tích nước thời kỳ thi công đối với công trình thủy lợi đập đất đầm nén được quy định như thế nào?
Relevant cids: ['65231']
BM25 pred parent_ids: ['237951', '224108', '65915', '105613', '13982', '65230', '140700', '74681', '65231', '14007']
Metrics: {'recall@1': 0.0, 'recall@3': 0.0, 'recall@5': 0.0, 'recall@10': 1.0, 'mrr': 0.1111111111111111, 'ndcg@10': 0.3010299956639812}

--- Sample 2 ---
Question: Phanh cần trục khi dừng khẩn cấp phải là loại phanh như thế nào?
Relevant cids: ['183501']
BM25 pred parent_ids: ['1

# Cell 10 - Đánh giá BM25 Retrieval trên validation set

## Mục tiêu cell này
Đánh giá chất lượng BM25 Retrieval trên toàn bộ `val_strict.csv`.

## Vì sao cần làm bước này?
Search thử vài câu chỉ cho biết BM25 hoạt động được, nhưng chưa đủ để kết luận chất lượng mô hình.  
Ta cần chạy trên toàn bộ validation set để đo các metric retrieval chính thức:

- Recall@1: tỷ lệ câu hỏi tìm đúng tài liệu ngay vị trí đầu tiên
- Recall@3: tỷ lệ câu hỏi tìm đúng tài liệu trong top 3
- Recall@5: tỷ lệ câu hỏi tìm đúng tài liệu trong top 5
- Recall@10: tỷ lệ câu hỏi tìm đúng tài liệu trong top 10
- MRR: tài liệu đúng xuất hiện càng sớm thì điểm càng cao
- nDCG@10: đánh giá chất lượng xếp hạng trong top 10

Validation set dùng để kiểm tra và tune pipeline, không dùng test set ở bước này.

## Giải thích code
Code sẽ:
1. Duyệt từng câu hỏi trong validation set
2. Dùng BM25 retrieve top 10 `parent_id`
3. So sánh với `relevant_cids`
4. Tính metric cho từng câu hỏi
5. Lấy trung bình metric toàn validation
6. Lưu kết quả chi tiết và kết quả tổng hợp vào `artifacts/metrics`

## Output mong đợi
Bạn cần thấy bảng metric tổng hợp của BM25 trên validation set.

In [12]:
from tqdm.auto import tqdm

METRIC_DIR = PROJECT / "artifacts" / "metrics"
PRED_DIR = PROJECT / "artifacts" / "predictions"

METRIC_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

bm25_val_rows = []

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    pred_ids = bm25_retrieve_parent_ids(row["question"], top_k=10)
    relevant_ids = [str(x) for x in row["relevant_cids"]]
    metrics = compute_retrieval_metrics(pred_ids, relevant_ids)

    bm25_val_rows.append({
        "qid": row["qid"],
        "question": row["question"],
        "relevant_cids": json.dumps(relevant_ids, ensure_ascii=False),
        "pred_parent_ids": json.dumps(pred_ids, ensure_ascii=False),
        **metrics
    })

bm25_val_results_df = pd.DataFrame(bm25_val_rows)

bm25_val_summary_df = (
    bm25_val_results_df[
        ["recall@1", "recall@3", "recall@5", "recall@10", "mrr", "ndcg@10"]
    ]
    .mean()
    .reset_index()
)

bm25_val_summary_df.columns = ["metric", "value"]
bm25_val_summary_df["method"] = "BM25"

results_path = PRED_DIR / "bm25_val_predictions.csv"
summary_path = METRIC_DIR / "bm25_val_metrics.csv"

bm25_val_results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
bm25_val_summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Đã lưu predictions:", results_path)
print("Đã lưu metrics:", summary_path)

display(bm25_val_summary_df)

  0%|          | 0/8605 [00:00<?, ?it/s]

Đã lưu predictions: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\predictions\bm25_val_predictions.csv
Đã lưu metrics: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\bm25_val_metrics.csv


,metric,value,method
0,recall@1,0.343754,BM25
1,recall@3,0.528995,BM25
2,recall@5,0.605113,BM25
3,recall@10,0.687391,BM25
4,mrr,0.453844,BM25
5,ndcg@10,0.497519,BM25


# Cell 11 - Phân tích ví dụ BM25 đúng và sai trên validation set

## Mục tiêu cell này
Xem một vài ví dụ BM25 truy xuất đúng và sai để hiểu rõ điểm mạnh, điểm yếu của BM25.

## Vì sao cần làm bước này?
Metric tổng hợp cho biết BM25 đạt bao nhiêu điểm, nhưng chưa giúp ta hiểu vì sao BM25 đúng hoặc sai.  
Trong báo cáo và thuyết trình, cần có ví dụ cụ thể để giải thích:

- Khi câu hỏi có từ khóa rõ, BM25 thường tìm đúng.
- Khi câu hỏi diễn đạt khác hoặc nhiều tài liệu có từ khóa giống nhau, BM25 có thể xếp sai.

## Giải thích code
Code sẽ:
1. Đọc kết quả `bm25_val_predictions.csv`
2. Tạo cột `hit@1` và `hit@10`
3. Lấy vài mẫu đúng ở top 1
4. Lấy vài mẫu sai trong top 10
5. Hiển thị câu hỏi, relevant cid và các cid BM25 dự đoán

## Output mong đợi
Bạn cần thấy 2 nhóm:
- Ví dụ BM25 đúng ngay top 1
- Ví dụ BM25 không tìm thấy tài liệu đúng trong top 10

In [13]:
bm25_val_results_df = pd.read_csv(PRED_DIR / "bm25_val_predictions.csv")

bm25_val_results_df["relevant_cids_list"] = bm25_val_results_df["relevant_cids"].apply(json.loads)
bm25_val_results_df["pred_parent_ids_list"] = bm25_val_results_df["pred_parent_ids"].apply(json.loads)

bm25_val_results_df["hit@1"] = bm25_val_results_df.apply(
    lambda row: int(row["pred_parent_ids_list"][0] in row["relevant_cids_list"]),
    axis=1
)

bm25_val_results_df["hit@10"] = bm25_val_results_df.apply(
    lambda row: int(bool(set(row["pred_parent_ids_list"]) & set(row["relevant_cids_list"]))),
    axis=1
)

correct_top1_df = bm25_val_results_df[bm25_val_results_df["hit@1"] == 1].head(5)
wrong_top10_df = bm25_val_results_df[bm25_val_results_df["hit@10"] == 0].head(5)

print("Số câu đúng ở top 1:", bm25_val_results_df["hit@1"].sum())
print("Số câu đúng trong top 10:", bm25_val_results_df["hit@10"].sum())
print("Số câu sai trong top 10:", len(bm25_val_results_df) - bm25_val_results_df["hit@10"].sum())

print("\nVí dụ BM25 đúng ngay top 1:")
display(correct_top1_df[["qid", "question", "relevant_cids", "pred_parent_ids", "recall@1", "mrr"]])

print("\nVí dụ BM25 sai trong top 10:")
display(wrong_top10_df[["qid", "question", "relevant_cids", "pred_parent_ids", "recall@10", "mrr"]])

Số câu đúng ở top 1: 2958
Số câu đúng trong top 10: 5915
Số câu sai trong top 10: 2690

Ví dụ BM25 đúng ngay top 1:


,qid,question,relevant_cids,pred_parent_ids,recall@1,mrr
2,108965,Phanh cần trục khi dừng khẩn cấp phải là loại ...,"[""183501""]","[""183501"", ""183502"", ""88619"", ""88621"", ""38369""...",1.0,1.0
5,161862,Quá 15 ngày kể từ ngày tuyên án thì có được kh...,"[""59639""]","[""59639"", ""10810"", ""72151"", ""47322"", ""10811"", ...",1.0,1.0
12,52533,Doanh nghiệp cho thuê lại lao động không lập h...,"[""112404""]","[""112404"", ""36506"", ""41342"", ""120311"", ""189335...",1.0,1.0
16,148197,Người cha có quyền yêu cầu xác định con không?,"[""47663""]","[""47663"", ""47650"", ""47662"", ""58155"", ""58152"", ...",1.0,1.0
17,7771,Ban Chỉ đạo cổ phần hóa đối với các doanh nghi...,"[""70144""]","[""70144"", ""70143"", ""21548"", ""205631"", ""70142"",...",1.0,1.0



Ví dụ BM25 sai trong top 10:


,qid,question,relevant_cids,pred_parent_ids,recall@10,mrr
10,76459,Khi sát hạch thực hành lái tàu trên đường sắt ...,"[""38367"", ""38367""]","[""38333"", ""123391"", ""38361"", ""82168"", ""38331"",...",0.0,0.0
15,37695,"Quyền của cơ quan, tổ chức, cá nhân chịu sự gi...","[""57027""]","[""108357"", ""238843"", ""59934"", ""197701"", ""10195...",0.0,0.0
18,18734,Người lao động cung cấp sai thông tin tài khoả...,"[""82415""]","[""189778"", ""67677"", ""122051"", ""107031"", ""11850...",0.0,0.0
19,44658,Quy định về mẫu dấu hợp quy đối với phương tiệ...,"[""83496""]","[""232999"", ""33062"", ""197701"", ""78903"", ""41554""...",0.0,0.0
20,136635,Ngân hàng thương mại kinh doanh bất động sản t...,"[""57545"", ""68706""]","[""60878"", ""235714"", ""156381"", ""201236"", ""79902...",0.0,0.0


# Cell 12 - Kiểm tra cuối các file BM25 đã tạo

## Mục tiêu cell này
Kiểm tra toàn bộ file đầu ra quan trọng của notebook `04_build_bm25_index.ipynb`.

## Vì sao cần làm bước này?
Notebook 04 đã xây dựng BM25 index và đánh giá BM25 trên validation set.  
Các notebook sau sẽ cần dùng lại các file này để:
- so sánh BM25 với Dense Retrieval
- kết hợp BM25 + Dense trong Hybrid Retrieval
- dùng BM25 predictions làm baseline
- viết bảng kết quả thực nghiệm

Nếu thiếu file, các bước sau sẽ lỗi hoặc không thể so sánh kết quả.

## Giải thích code
Code sẽ kiểm tra sự tồn tại của:
- `bm25_index.pkl`: index BM25 đã build
- `bm25_metadata.csv`: metadata của toàn bộ chunks
- `bm25_val_predictions.csv`: kết quả BM25 trên validation
- `bm25_val_metrics.csv`: metric BM25 trên validation

Sau đó in thêm metric validation để xác nhận baseline BM25 đã lưu đúng.

## Output mong đợi
Tất cả file cần có trạng thái `OK`.  
Bảng metric BM25 validation phải hiển thị Recall@1, Recall@3, Recall@5, Recall@10, MRR và nDCG@10.

In [14]:
required_bm25_files = [
    BM25_DIR / "bm25_index.pkl",
    BM25_DIR / "bm25_metadata.csv",
    PRED_DIR / "bm25_val_predictions.csv",
    METRIC_DIR / "bm25_val_metrics.csv"
]

bm25_check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0
    }
    for path in required_bm25_files
])

bm25_val_metrics_df = pd.read_csv(METRIC_DIR / "bm25_val_metrics.csv")

display(bm25_check_df)
display(bm25_val_metrics_df)

print("Tổng file OK:", (bm25_check_df["status"] == "OK").sum(), "/", len(bm25_check_df))

,file,status,size_mb
0,indexes\bm25\bm25_index.pkl,OK,71.46
1,indexes\bm25\bm25_metadata.csv,OK,182.62
2,artifacts\predictions\bm25_val_predictions.csv,OK,2.42
3,artifacts\metrics\bm25_val_metrics.csv,OK,0.00


,metric,value,method
0,recall@1,0.343754,BM25
1,recall@3,0.528995,BM25
2,recall@5,0.605113,BM25
3,recall@10,0.687391,BM25
4,mrr,0.453844,BM25
5,ndcg@10,0.497519,BM25


Tổng file OK: 4 / 4




## 1. File 04 làm gì?

File 04 tạo một **công cụ tìm kiếm theo từ khóa** cho toàn bộ kho tài liệu RAG.

Kho tài liệu của mình hiện là:

```text
data/chunks/all_chunks.csv
```

Trong đó có:

```text
91,353 chunks pháp luật Việt Nam
95 chunks handbook nội bộ công ty
Tổng: 91,448 chunks
```

File 04 lấy toàn bộ chunks này và build **BM25 index**.

Nói dễ hiểu:

```text
File 03 tạo “kho tài liệu đã cắt nhỏ”.
File 04 tạo “máy tìm kiếm từ khóa” trên kho tài liệu đó.
```

---

## 2. BM25 là gì trong bài mình?

BM25 giống như công cụ search trong Google nhưng đơn giản hơn. Nó tìm tài liệu dựa trên **từ khóa trùng hoặc liên quan trực tiếp**.

Ví dụ người dùng hỏi:

```text
Lương thử việc được quy định như thế nào?
```

BM25 sẽ ưu tiên các chunk có các từ như:

```text
lương
thử việc
quy định
người lao động
```

Rồi trả về danh sách tài liệu/chunk liên quan nhất.

---

## 3. Ví dụ thực tế trong app của mình

Giả sử nhân viên hỏi chatbot:

```text
Công ty trả lương thử việc 70% có ổn không?
```

Hệ thống cần tìm tài liệu liên quan trong:

```text
1. Handbook nội bộ công ty
2. Văn bản pháp luật Việt Nam
```

BM25 sẽ tìm theo từ khóa như:

```text
công ty
lương
thử việc
70%
```

Nếu BM25 tìm được đúng chunk pháp luật về lương thử việc, thì LLM có nguồn để trả lời.

Nếu BM25 không tìm được đúng chunk, LLM sẽ thiếu căn cứ và có thể phải từ chối hoặc trả lời kém.

---

## 4. File 04 đã làm những bước nào?

### Bước 1: Load corpus

Đọc file:

```text
data/chunks/all_chunks.csv
```

Đây là kho bằng chứng gồm legal + handbook.

---

### Bước 2: Chuẩn hóa dữ liệu

Vì `parent_id` có cả số và chữ:

```text
Legal parent_id: 142820
Company parent_id: company_0000
```

nên mình ép toàn bộ về chuỗi để tránh lỗi.

---

### Bước 3: Tokenize

BM25 không hiểu nguyên câu, nên phải tách câu thành token.

Ví dụ:

```text
Công ty có chính sách gì về thiết bị làm việc?
```

thành:

```text
["công", "ty", "có", "chính", "sách", "gì", "về", "thiết", "bị", "làm", "việc"]
```

---

### Bước 4: Build BM25 index

Mình dùng `rank_bm25` để tạo BM25 index từ 91,448 chunks.

Kết quả đã lưu:

```text
indexes/bm25/bm25_index.pkl
indexes/bm25/bm25_metadata.csv
```

---

### Bước 5: Search thử

Bạn đã test 2 câu:

```text
Lương thử việc được quy định như thế nào?
```

BM25 trả về các chunk pháp luật liên quan đến:

```text
mức lương
trả lương
người thử việc
bảo hiểm đối với người thử việc
```

Và câu:

```text
What is the company policy for managing work devices?
```

BM25 tìm được chunk:

```text
Managing Work Devices
```

Điều đó chứng minh BM25 search hoạt động.

---

## 5. File 04 đánh giá BM25 như thế nào?

Mình dùng validation set có:

```text
8,605 câu hỏi
```

Mỗi câu hỏi có sẵn `relevant_cids`, tức là tài liệu đúng.

BM25 trả về top 10 tài liệu dự đoán. Sau đó mình so với tài liệu đúng.

Ví dụ:

```text
Câu hỏi: Phanh cần trục khi dừng khẩn cấp phải là loại phanh như thế nào?
Relevant cid đúng: 183501

BM25 trả về:
Rank 1: 183501 ✅
Rank 2: 183502
Rank 3: 88619
...
```

Vì tài liệu đúng nằm ở rank 1 nên:

```text
Recall@1 = 1
MRR = 1
```

Ví dụ sai:

```text
Relevant cid đúng: 57545, 68706

BM25 top 10:
60878, 235714, 156381, ...
```

Không có cid đúng trong top 10 nên:

```text
Recall@10 = 0
MRR = 0
```

---

## 6. Kết quả BM25 của bạn nói gì?

Kết quả cuối:

```text
Recall@1  = 0.3438
Recall@3  = 0.5290
Recall@5  = 0.6051
Recall@10 = 0.6874
MRR       = 0.4538
nDCG@10   = 0.4975
```

Hiểu đơn giản:

```text
Trong 100 câu hỏi:
- khoảng 34 câu BM25 đưa đúng tài liệu lên top 1
- khoảng 60 câu BM25 tìm đúng tài liệu trong top 5
- khoảng 68 câu BM25 tìm đúng tài liệu trong top 10
```

Nghĩa là BM25 **có ích**, nhưng chưa đủ mạnh.

---

## 7. Vì sao BM25 chưa đủ?

BM25 tìm theo từ khóa. Nếu người dùng hỏi khác cách diễn đạt, BM25 có thể yếu.

Ví dụ tài liệu ghi:

```text
quản lý thiết bị làm việc
```

Người dùng hỏi:

```text
laptop công ty khi nghỉ việc xử lý sao?
```

BM25 có thể không bắt tốt vì từ khóa khác nhau.

Vì vậy các file sau sẽ làm:

```text
05_build_dense_index.ipynb
→ tìm kiếm theo ngữ nghĩa

07_hybrid_rag.ipynb
→ kết hợp BM25 + Dense

08_reranker_rag.ipynb
→ đọc lại top kết quả và xếp hạng chính xác hơn
```

---

## 8. Chốt lại file 04

File 04 tạo baseline quan trọng:

```text
BM25 = keyword retrieval baseline
```

Nó giúp mình chứng minh:

```text
Chỉ tìm kiếm từ khóa thì đạt Recall@10 khoảng 68.7%.
Đề tài cần Dense Retrieval, Hybrid Retrieval và Reranker để cải thiện.
```

Nói ngắn gọn:

```text
File 04 xây máy tìm kiếm từ khóa đầu tiên cho RAG và đo xem nó tìm tài liệu đúng tốt đến đâu.
```
